# SDC New Issues — Equity & Debt Issuance EDA

Annual counts of US securities offerings from SDC New Issues and FISD/Mergent.

**Data sources:**
- SDC Global New Issues (`tdc1.sdc_ni` — schema must be discovered first)
- FISD/Mergent (`fisd.fisd_mergedissue`) for registered + 144A corporate bonds

**Reference:** `references/sdc-issuances.md`, `references/fisd-bonds.md`

## Contents
1. Schema discovery (SDC)
2. SDC equity issuances: IPOs vs SEOs, annual counts
3. SDC debt issuances: annual count by security type
4. FISD bond issuances: IG vs HY, 144A vs registered
5. Combined visualization: capital markets practice area by offering type

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

conn = psycopg2.connect(
    host='wrds-pgdata.wharton.upenn.edu',
    port=9737,
    database='wrds',
    user='eddyhu',
    sslmode='require'
)
print('Connected to WRDS')

## 1. Schema Discovery

In [ ]:
cur = conn.cursor()

# Find SDC-related schemas
cur.execute("""
    SELECT schema_name
    FROM information_schema.schemata
    WHERE schema_name ILIKE '%sdc%'
       OR schema_name ILIKE '%tdc%'
       OR schema_name ILIKE '%deals%'
    ORDER BY schema_name
""")
schemas = cur.fetchall()
print('SDC-related schemas:', schemas)

# Set schema from discovery results (update this variable)
SDC_SCHEMA = schemas[0][0] if schemas else 'tdc1'
print(f'Using schema: {SDC_SCHEMA}')

In [ ]:
# Find new issues table within schema
cur.execute("""
    SELECT table_name,
           pg_size_pretty(pg_total_relation_size(
               quote_ident(table_schema)||'.'||quote_ident(table_name))) AS size
    FROM information_schema.tables
    WHERE table_schema = %s
    ORDER BY table_name
""", (SDC_SCHEMA,))
tables = cur.fetchall()
print('Tables in schema:')
for t in tables:
    print(' ', t)

# Set new issues table name (update based on output above)
NI_TABLE = f'{SDC_SCHEMA}.sdc_ni'  # confirm from discovery

In [ ]:
# Inspect new issues columns
ni_table_name = NI_TABLE.split('.')[-1]
cur.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = %s AND table_name = %s
    ORDER BY ordinal_position
""", (SDC_SCHEMA, ni_table_name))
cols = cur.fetchall()
print(f'Columns in {NI_TABLE}:')
for c in cols:
    print(f'  {c[0]:40s} {c[1]}')

In [ ]:
# Row count and date range check
cur.execute(f"""
    SELECT
        COUNT(*)                      AS total_rows,
        COUNT(DISTINCT deal_no)       AS unique_deals,
        MIN(issue_date)               AS earliest,
        MAX(issue_date)               AS latest
    FROM {NI_TABLE}
""")
row = cur.fetchone()
print(f'Total rows:     {row[0]:,.0f}')
print(f'Unique deals:   {row[1]:,.0f}')
print(f'Date range:     {row[2]} → {row[3]}')

## 2. SDC Equity Issuances: IPOs vs SEOs

In [ ]:
# Pull US common stock new issues 1985-present
# Column names depend on schema discovery above — adjust if needed
query_equity = f"""
SELECT
    deal_no,
    issue_date,
    EXTRACT(YEAR FROM issue_date)::int AS issue_year,
    issuer,
    nation,
    ipo              AS ipo_flag,
    orig_ipo         AS orig_ipo_flag,
    offer_price,
    proceeds,
    security_type,
    vc               AS vc_backed,
    reit,
    adr,
    closed_end_fund,
    unit,
    cusip9
FROM {NI_TABLE}
WHERE nation = 'United States'
  AND issue_date BETWEEN '1985-01-01' AND '2024-12-31'
  AND proceeds IS NOT NULL
  AND proceeds > 0
"""
df_eq = pd.read_sql(query_equity, conn)
print(f'Raw US equity rows: {len(df_eq):,}')
df_eq.head(3)

In [ ]:
# Apply standard cleaning filters
EXCLUDE_TYPES = {
    'Units', 'Ltd Prtnr Int', 'MLP-Common Shs',
    'Shs Benficl Int', 'Ltd Liab Int', 'Stock Unit',
    'Trust Units', 'Beneficial Ints'
}

# IPOs: both IPO flags must not be 'No'
df_ipo = df_eq[
    (df_eq['ipo_flag'] != 'No') &
    (df_eq['orig_ipo_flag'].fillna('Yes') != 'No') &
    (~df_eq['security_type'].isin(EXCLUDE_TYPES)) &
    (df_eq['reit'].isna() | (df_eq['reit'] == '')) &
    (df_eq['adr'] == 'No') &
    ((df_eq['closed_end_fund'] == 'No') | df_eq['closed_end_fund'].isna()) &
    ((df_eq['unit'] == 'No') | (df_eq['unit'] == '') | df_eq['unit'].isna()) &
    (pd.to_numeric(df_eq['offer_price'], errors='coerce') >= 5.0)
].copy()

# SEOs: IPO flag = 'No', common stock
df_seo = df_eq[
    (df_eq['ipo_flag'] == 'No') &
    (~df_eq['security_type'].isin(EXCLUDE_TYPES)) &
    (df_eq['adr'] == 'No') &
    (pd.to_numeric(df_eq['offer_price'], errors='coerce') >= 1.0)
].copy()

print(f'Clean IPOs:  {len(df_ipo):,}')
print(f'Clean SEOs:  {len(df_seo):,}')

In [ ]:
# Annual IPO and SEO counts
ipo_annual = df_ipo.groupby('issue_year').agg(
    n_ipos=('deal_no', 'count'),
    proceeds_bn=('proceeds', lambda x: x.sum() / 1e3)
)
seo_annual = df_seo.groupby('issue_year').agg(
    n_seos=('deal_no', 'count'),
    proceeds_bn=('proceeds', lambda x: x.sum() / 1e3)
)

equity_annual = ipo_annual.join(seo_annual, how='outer', lsuffix='_ipo', rsuffix='_seo').fillna(0)
equity_annual = equity_annual[equity_annual.index.between(1985, 2024)]
print(equity_annual.tail(10))

In [ ]:
# Plot IPO vs SEO counts
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax0, ax1 = axes
ax0.bar(equity_annual.index, equity_annual['n_ipos'], color='steelblue',
        label='IPOs', alpha=0.85)
ax0.bar(equity_annual.index, equity_annual['n_seos'], bottom=0,
        color='coral', alpha=0.6, label='SEOs')
ax0.set_ylabel('Number of Offerings')
ax0.set_title('US Equity Issuances: IPOs vs SEOs (SDC New Issues)')
ax0.legend()

ax1.bar(equity_annual.index, equity_annual['proceeds_bn_ipo'],
        color='steelblue', label='IPO Proceeds', alpha=0.85)
ax1.bar(equity_annual.index, equity_annual['proceeds_bn_seo'],
        bottom=0, color='coral', alpha=0.6, label='SEO Proceeds')
ax1.set_ylabel('Proceeds ($Bn)')
ax1.set_xlabel('Year')
ax1.legend()

plt.tight_layout()
plt.savefig('/tmp/equity_issuances.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /tmp/equity_issuances.png')

## 3. FISD Bond Issuances: IG vs HY, 144A vs Registered

In [ ]:
# FISD: annual US corporate bond issuances
query_bonds = """
SELECT
    EXTRACT(YEAR FROM i.offering_date)::int  AS issue_year,
    i.rule_144a,
    CASE
        WHEN i.moody_rating IN ('Aaa','Aa1','Aa2','Aa3',
                                'A1','A2','A3','Baa1','Baa2','Baa3')
          OR i.sp_rating IN ('AAA','AA+','AA','AA-',
                              'A+','A','A-','BBB+','BBB','BBB-')
        THEN 'IG'
        WHEN i.moody_rating IN ('Ba1','Ba2','Ba3','B1','B2','B3',
                                'Caa1','Caa2','Caa3','Ca','C')
          OR i.sp_rating IN ('BB+','BB','BB-','B+','B','B-',
                              'CCC+','CCC','CCC-','CC','C','D')
        THEN 'HY'
        ELSE 'NR'
    END                                      AS rating_cat,
    COUNT(*)                                 AS n_issues,
    SUM(i.offering_amt) / 1e3               AS proceeds_bn
FROM fisd.fisd_mergedissue i
JOIN fisd.fisd_mergedissuer u ON i.issuer_id = u.issuer_id
WHERE u.country_domicile = 'USA'
  AND i.bond_type IN ('CDEB','CMTN','CMTZ','CZ','USBN')
  AND (i.yankee = 'N' OR i.yankee IS NULL)
  AND (i.canadian = 'N' OR i.canadian IS NULL)
  AND (i.foreign_currency = 'N' OR i.foreign_currency IS NULL)
  AND (i.asset_backed = 'N' OR i.asset_backed IS NULL)
  AND (i.convertible = 'N' OR i.convertible IS NULL)
  AND (i.preferred_security = 'N' OR i.preferred_security IS NULL)
  AND (i.defeased = 'N' OR i.defeased IS NULL)
  AND i.offering_date BETWEEN '1990-01-01' AND '2024-12-31'
  AND i.offering_amt > 0
GROUP BY issue_year, i.rule_144a, rating_cat
ORDER BY issue_year, rating_cat, i.rule_144a
"""
df_bonds = pd.read_sql(query_bonds, conn)
print(f'Bond issuance rows: {len(df_bonds):,}')
df_bonds.head(8)

In [ ]:
# Pivot to wide format: IG/HY × 144A/Reg
df_bonds['category'] = (
    df_bonds['rating_cat'] + '_' +
    df_bonds['rule_144a'].map({'Y': '144A', 'N': 'Reg'}).fillna('Reg')
)

bond_pivot = (df_bonds
    .groupby(['issue_year', 'category'])[['n_issues', 'proceeds_bn']]
    .sum()
    .unstack('category')
    .fillna(0)
)
bond_pivot = bond_pivot[bond_pivot.index.between(1995, 2024)]
print('Categories:', bond_pivot.columns.get_level_values(1).unique().tolist())
print(bond_pivot['n_issues'].tail(5))

In [ ]:
# Plot: bond issuances by IG/HY and 144A/Reg
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count
ax = axes[0]
n = bond_pivot['n_issues']
cats = [c for c in ['IG_Reg','IG_144A','HY_Reg','HY_144A'] if c in n.columns]
colors = ['#1f77b4','#aec7e8','#d62728','#f7b6d2']
bottom = np.zeros(len(n))
for cat, col in zip(cats, colors):
    if cat in n.columns:
        ax.bar(n.index, n[cat], bottom=bottom, label=cat, color=col, alpha=0.9)
        bottom += n[cat].values
ax.set_title('US Bond Issuances: Count')
ax.set_ylabel('Number of Issuances')
ax.set_xlabel('Year')
ax.legend(fontsize=8)

# Volume
ax = axes[1]
v = bond_pivot['proceeds_bn']
bottom = np.zeros(len(v))
for cat, col in zip(cats, colors):
    if cat in v.columns:
        ax.bar(v.index, v[cat], bottom=bottom, label=cat, color=col, alpha=0.9)
        bottom += v[cat].values
ax.set_title('US Bond Issuances: Volume ($Bn)')
ax.set_ylabel('Proceeds ($Bn)')
ax.set_xlabel('Year')

plt.tight_layout()
plt.savefig('/tmp/bond_issuances.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /tmp/bond_issuances.png')

## 4. 144A Share of Bond Market Over Time

In [ ]:
# 144A share by rating category
for rc in ['IG', 'HY']:
    reg_col = f'{rc}_Reg'
    a44_col = f'{rc}_144A'
    n = bond_pivot['n_issues']
    if reg_col in n.columns and a44_col in n.columns:
        total = n[reg_col] + n[a44_col]
        share_144a = (n[a44_col] / total.replace(0, np.nan) * 100).fillna(0)
        print(f'{rc} 144A share (recent years):')
        print(share_144a.tail(10).round(1))
        print()

## 5. Summary: Capital Markets Practice Area Sizing

In [ ]:
# Recent 5-year average deal counts (practice area proxy)
recent_years = range(2018, 2023)

ipos_avg = equity_annual.loc[equity_annual.index.isin(recent_years), 'n_ipos'].mean()
seos_avg = equity_annual.loc[equity_annual.index.isin(recent_years), 'n_seos'].mean()

n = bond_pivot['n_issues']
ig_144a_avg = n.loc[n.index.isin(recent_years), 'IG_144A'].mean() if 'IG_144A' in n.columns else 0
hy_144a_avg = n.loc[n.index.isin(recent_years), 'HY_144A'].mean() if 'HY_144A' in n.columns else 0
ig_reg_avg  = n.loc[n.index.isin(recent_years), 'IG_Reg'].mean()  if 'IG_Reg'  in n.columns else 0
hy_reg_avg  = n.loc[n.index.isin(recent_years), 'HY_Reg'].mean()  if 'HY_Reg'  in n.columns else 0

summary = pd.DataFrame({
    'Practice Area': ['IPOs', 'SEOs (Follow-ons)', 'IG Bonds (144A)',
                      'HY Bonds (144A)', 'IG Bonds (Registered)', 'HY Bonds (Registered)'],
    'Avg Annual Deals (2018–22)': [
        int(ipos_avg), int(seos_avg),
        int(ig_144a_avg), int(hy_144a_avg),
        int(ig_reg_avg), int(hy_reg_avg)
    ]
})
summary = summary.sort_values('Avg Annual Deals (2018–22)', ascending=False)
print(summary.to_string(index=False))

In [ ]:
# Horizontal bar chart: capital markets practice area by deal count
fig, ax = plt.subplots(figsize=(10, 5))
colors_map = {
    'IPOs': '#1f77b4',
    'SEOs (Follow-ons)': '#aec7e8',
    'IG Bonds (144A)': '#2ca02c',
    'HY Bonds (144A)': '#d62728',
    'IG Bonds (Registered)': '#98df8a',
    'HY Bonds (Registered)': '#f7b6d2',
}
ax.barh(
    summary['Practice Area'],
    summary['Avg Annual Deals (2018–22)'],
    color=[colors_map.get(pa, 'gray') for pa in summary['Practice Area']]
)
ax.set_xlabel('Average Annual Deal Count (2018–2022)')
ax.set_title('Capital Markets Practice Area Sizing\n(US, 2018–2022 Average)')
for i, (_, row) in enumerate(summary.iterrows()):
    ax.text(row['Avg Annual Deals (2018–22)'] + 5, i,
            f"{row['Avg Annual Deals (2018–22)']:,}", va='center', fontsize=9)
ax.invert_yaxis()

plt.tight_layout()
plt.savefig('/tmp/capital_markets_sizing.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /tmp/capital_markets_sizing.png')
print()
print('KEY FINDING: 144A bond issuances (IG + HY) dwarf IPOs and SEOs in deal count.')
print('The typical capital markets lawyer does far more bond/144A work than IPO work.')